# Phase 3 — Architecture 3: Legal Prefix–Conv LM

In [1]:
%pip -q install "peft==0.17.1" "transformers==4.55.4" "accelerate==1.10.1" "tokenizers==0.21.4" "huggingface_hub==0.34.4" "cloudpickle==3.1.1" scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.

In [2]:
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil, ast, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, classification_report
from peft import LoraConfig, IA3Config, LoHaConfig, PromptTuningConfig, inject_adapter_in_model
from peft.tuners.prompt_tuning import PromptEmbedding

BASE = Path('/kaggle/input/datasets/kumaralpha1/a3-p3')
OUT = Path('/kaggle/working/phase3_a3_peft')

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, target=320,
         micro=2, accum=8, eval_batch=8, rank=16, prompt=16,
         eval_steps=200, patience=5, classes=13)
METHODS = {
    'full': dict(epochs=6, lr=2e-5),
    'partial': dict(epochs=8, lr=5e-5),
    'lora': dict(epochs=10, lr=2e-4),
    'lorma': dict(epochs=10, lr=2e-4),
    'ia3': dict(epochs=10, lr=5e-4),
    'loha': dict(epochs=10, lr=2e-4),
    'prompt': dict(epochs=12, lr=3e-3),
    'distill': dict(epochs=8, lr=4e-5),
}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])

## Dataset and sentence windows

In [3]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = ['Train/CL_train.json', 'Train/CL_dev.json', 'Train/IT_train.json',
         'Train/IT_dev.json', 'Train/label_vocab.json', 'Test/CL_test.json', 'Test/IT_test.json']
raw = {}
for name in files:
    p = hf_hub_download(REPO, 'Phase_3_RR/'+name, repo_type='dataset',
                        revision=revision, token=HF_TOKEN)
    raw[Path(name).stem] = json.loads(Path(p).read_text())
vocab = raw['label_vocab']
if isinstance(vocab, dict) and 'label2id' in vocab:
    vocab = vocab['label2id']
if isinstance(vocab, dict) and 'id2label' in vocab:
    vocab = vocab['id2label']
if isinstance(vocab, list):
    labels = vocab
elif all(str(k).isdigit() for k in vocab):
    labels = [vocab[k] for k in sorted(vocab, key=lambda k: int(k))]
else:
    labels = sorted(vocab, key=lambda k: int(vocab[k]))
label2id = {s:i for i,s in enumerate(labels)}
R['classes'] = len(labels)

def documents(data, domain, labelled):
    if isinstance(data, dict) and 'id' in data and 'text' in data:
        rows = [data]
    elif isinstance(data, dict):
        rows = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v)
                for k,v in data.items()]
    else:
        rows = data
    docs = []
    for row in rows:
        text = row['text']
        if not isinstance(text, list) or any(not isinstance(s, str) for s in text):
            raise ValueError('Expected an ordered list of sentence strings in text')
        y = row['labels'] if labelled else []
        y = [label2id[v] if isinstance(v, str) and v in label2id else int(v) for v in y]
        docs.append(dict(id=str(row['id']), text=text, y=y, domain=domain))
    return docs

DOCS = {}
for split in ['train','dev','test']:
    DOCS[split] = sum([documents(raw[f'{domain}_{split}'], d, split!='test')
                       for d,domain in enumerate(['CL','IT'])], [])

tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
bos = tok.token_to_id('<bos>')
pad = tok.token_to_id('<pad>')
R['pad'] = pad

def encode_docs(docs):
    rows = []
    for doc in docs:
        parts = [e.ids for e in tok.encode_batch(doc['text'])]
        for i,ids in enumerate(parts):
            if len(ids) > R['target']:
                half = R['target']//2
                ids = ids[:half] + ids[-(R['target']-half):]
            previous = sum([p+[bos] for p in parts[max(0,i-3):i]], [])
            budget = R['length']-len(ids)-2
            context = previous[-budget:] if budget else []
            x = context+[bos]+ids+[bos]
            rows.append(dict(x=x, start=len(context)+1, end=len(x)-1,
                             y=doc['y'][i] if doc['y'] else -100,
                             domain=doc['domain'], pos=i/max(1,len(parts)-1),
                             id=doc['id'], sentence=i))
    return rows

DATA = {s:encode_docs(ds) for s,ds in DOCS.items()}
with (OUT/'data.pkl').open('wb') as f:
    cloudpickle.dump(dict(rows=DATA, docs=DOCS, labels=labels), f)
R['run_id'] = hashlib.sha256(json.dumps([C,{k:v for k,v in R.items() if k!='run_id'},METHODS,labels,revision],sort_keys=True).encode()).hexdigest()
LORMA_COMMIT = 'ffb3c1de29a7f2ab880dc504a2e473b5a4b63292'
LORMA_URL = f'https://raw.githubusercontent.com/Exploration-Lab/LoRMA/{LORMA_COMMIT}/NLU/run_glue_lorma_plus.py'
lorma_source = urllib.request.urlopen(LORMA_URL).read().decode()
(OUT/'lorma_source.txt').write_text(lorma_source)
manifest = dict(architecture='Architecture 3 Prefix-Conv', attention='bidirectional prefix, valid keys only', student_layers=6, lorma_commit=LORMA_COMMIT, peft_version='0.17.1', repo=REPO, revision=revision, config=C, settings=R, methods=METHODS,
                labels=labels, files=files,
                tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest())
(OUT/'manifest.json').write_text(json.dumps(manifest, indent=2))
shutil.copyfile(BASE/'tokenizer.json', OUT/'tokenizer.json')
print(' | '.join(f'{s}: {len(ds)} documents, {len(DATA[s]):,} sentences' for s,ds in DOCS.items()))

CL_train.json: 0.00B [00:00, ?B/s]

CL_dev.json: 0.00B [00:00, ?B/s]

IT_train.json: 0.00B [00:00, ?B/s]

IT_dev.json: 0.00B [00:00, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json: 0.00B [00:00, ?B/s]

IT_test.json: 0.00B [00:00, ?B/s]

train: 80 documents, 16,845 sentences | dev: 10 documents, 2,142 sentences | test: 10 documents, 2,197 sentences


## Architecture 3 — Legal Prefix–Conv LM

In [4]:
def norm(x, w):
    y = x.float()
    return (y*torch.rsqrt(y.square().mean(-1,keepdim=True)+1e-6)*w).to(x.dtype)

def rope(x, cos, sin):
    a,b = x.float().chunk(2,-1)
    return torch.cat((a*cos-b*sin,b*cos+a*sin),-1).to(x.dtype)

class PrefixBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        d,f = c['width'],c['ff']
        self.h = c['heads']
        self.p = nn.ParameterDict({k:nn.Parameter(torch.ones(d)) for k in ['n1','n2','nc']})
        self.p['conv'] = nn.Parameter(torch.empty(3,d))
        self.p['gate'] = nn.Parameter(torch.zeros(d))
        self.q_proj,self.k_proj,self.v_proj = [nn.Linear(d,d,bias=False) for _ in range(3)]
        self.out = nn.Linear(d,d,bias=False)
        self.up = nn.Linear(d,2*f,bias=False)
        self.down = nn.Linear(f,d,bias=False)
    def forward(self, x, mask, cos, sin):
        p = self.p
        u = norm(x,p['nc'])
        z = F.pad(u,(0,0,2,0))
        local = sum(z[:,2-i:2-i+x.size(1)]*p['conv'][i].to(x.dtype) for i in range(3))
        x = x+torch.sigmoid(u+p['gate'].to(x.dtype))*local
        u = norm(x,p['n1'])
        q,k,v = [layer(u).view(x.size(0),x.size(1),self.h,-1).transpose(1,2)
                 for layer in [self.q_proj,self.k_proj,self.v_proj]]
        q,k = rope(q,cos,sin),rope(k,cos,sin)
        a = F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
        x = x+self.out(a.transpose(1,2).reshape_as(x))
        a,b = self.up(norm(x,p['n2'])).chunk(2,-1)
        return x+self.down(F.silu(a)*b)

class PrefixBackbone(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.embed = nn.Parameter(torch.empty(c['vocab'],c['width']))
        self.final = nn.Parameter(torch.ones(c['width']))
        self.blocks = nn.ModuleList([PrefixBlock(c) for _ in range(c['layers'])])
        half = c['width']//c['heads']//2
        angle = torch.arange(c['prefix']+1024).float()[:,None]*10000.**(-torch.arange(half).float()/half)
        self.register_buffer('cos',angle.cos()[None,None],persistent=False)
        self.register_buffer('sin',angle.sin()[None,None],persistent=False)

In [5]:
tree = ast.parse((OUT/'lorma_source.txt').read_text())
node = next(v for v in tree.body if isinstance(v,ast.ClassDef) and v.name=='LormaLinear_plus')
namespace = dict(torch=torch, nn=nn, F=F, math=math, __name__='__main__')
exec(compile(ast.Module(body=[node],type_ignores=[]),'<LORMA authors implementation>','exec'),namespace)
LormaLinear = namespace['LormaLinear_plus']

class RoleModel(nn.Module):
    def __init__(self, lm, method, r):
        super().__init__()
        self.lm = lm
        d = lm.c['width']
        self.domain = nn.Embedding(2,8)
        self.head = nn.Sequential(nn.Dropout(.15), nn.Linear(2*d+9,256), nn.GELU(),
                                  nn.Dropout(.15), nn.Linear(256,r['classes']))
        self.prompt = None
        if method=='prompt':
            config = PromptTuningConfig(num_virtual_tokens=r['prompt'],token_dim=d,
                                        num_transformer_submodules=1)
            self.prompt = PromptEmbedding(config,None)
            self.adapter_config = config
    def forward(self, x, start, end, domain, pos):
        h = F.embedding(x,self.lm.embed)
        if torch.is_autocast_enabled():
            h = h.to(torch.get_autocast_dtype(h.device.type))
        shift = 0
        if self.prompt is not None:
            shift = self.prompt.embedding.num_embeddings
            tokens = torch.arange(shift,device=x.device)[None].expand(x.size(0),-1)
            h = torch.cat((self.prompt(tokens).to(h.dtype),h),1)
        start,end = start+shift,end+shift
        length = h.size(1)
        mask = torch.arange(length,device=x.device)[None,None,None,:]<=end[:,None,None,None]
        cos,sin = self.lm.cos[:,:,:length],self.lm.sin[:,:,:length]
        for block in self.lm.blocks:
            h = checkpoint(block,h,mask,cos,sin,use_reentrant=False) if self.training else block(h,mask,cos,sin)
        h = norm(h,self.lm.final)
        t = torch.arange(h.size(1),device=h.device)[None]
        mask = (t>=start[:,None]) & (t<end[:,None])
        mean = (h*mask[:,:,None]).sum(1)/mask.sum(1).clamp_min(1)[:,None]
        last = h[torch.arange(h.size(0),device=h.device),end]
        return self.head(torch.cat((mean,last,self.domain(domain),pos[:,None]),-1))

def make_model(method, c, r):
    saved = torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
    config = saved.get('config',c).copy()
    state = saved['model']
    if method=='distill':
        chosen = np.linspace(2,config['layers']-1,6,dtype=int).tolist()
        config['layers'] = 6
        state = {**{k:v for k,v in state.items() if not k.startswith('blocks.')},
                 **{f'blocks.{i}.'+k.split('.',2)[2]:v for i,j in enumerate(chosen)
                    for k,v in state.items() if k.startswith(f'blocks.{j}.')}}
    lm = PrefixBackbone(config)
    converted = {k:v for k,v in state.items() if k in ['embed','final'] or
                 (k.startswith('blocks.') and k.split('.')[-1] in ['n1','n2','nc','conv','gate'])}
    for i in range(config['layers']):
        prefix = f'blocks.{i}.'
        for name,w in zip(['q_proj','k_proj','v_proj'],state[prefix+'p.qkv'].chunk(3,1)):
            converted[prefix+name+'.weight'] = w.T.contiguous()
        for name in ['out','up','down']:
            converted[prefix+name+'.weight'] = state[prefix+'p.'+name].T.contiguous()
    lm.load_state_dict(converted,strict=True)
    del saved,state,converted
    if method not in ['full','distill']:
        lm.requires_grad_(False)
    if method=='partial':
        for b in lm.blocks[-6:]: b.requires_grad_(True)
        lm.final.requires_grad_(True)
    targets = ['q_proj','v_proj','out','up','down']
    adapter_config = None
    if method=='lora':
        adapter_config = LoraConfig(r=r['rank'],lora_alpha=2*r['rank'],lora_dropout=.05,
                                    target_modules=targets,bias='none')
    if method=='ia3':
        adapter_config = IA3Config(target_modules=['k_proj','v_proj','down'],
                                   feedforward_modules=['down'])
    if method=='loha':
        adapter_config = LoHaConfig(r=8,alpha=16,target_modules=['q_proj','v_proj','out'])
    if adapter_config is not None:
        lm = inject_adapter_in_model(adapter_config,lm)
    if method=='lorma':
        for name,layer in list(lm.named_modules()):
            if isinstance(layer,nn.Linear) and name.split('.')[-1] in targets:
                parent,key = name.rsplit('.',1)
                setattr(lm.get_submodule(parent),key,LormaLinear(layer,r['rank'],2*r['rank'],'pre',True,.05))
    model = RoleModel(lm,method,r)
    if adapter_config is not None:
        model.adapter_config = adapter_config
    return model

def collate(rows, indices, device, r):
    selected = [rows[int(i)] for i in indices]
    length = math.ceil(max(len(v['x']) for v in selected)/8)*8
    x = torch.full((len(selected),length),r['pad'],dtype=torch.long)
    for i,v in enumerate(selected): x[i,:len(v['x'])] = torch.tensor(v['x'])
    values = [x] + [torch.tensor([v[k] for v in selected],dtype=dtype)
                      for k,dtype in [('start',torch.long),('end',torch.long),
                                      ('domain',torch.long),('pos',torch.float32),('y',torch.long)]]
    return [v.pin_memory().to(device,non_blocking=True) for v in values]

@torch.inference_mode()
def predict(model, rows, device, r):
    model.eval()
    logits = np.empty((len(rows),r['classes']),dtype=np.float32)
    order = sorted(range(len(rows)),key=lambda i:len(rows[i]['x']))
    for at in range(0,len(order),r['eval_batch']):
        ix = order[at:at+r['eval_batch']]
        x,s,e,d,p,y = collate(rows,ix,device,r)
        with torch.autocast('cuda',dtype=torch.float16):
            z = model(x,s,e,d,p)
        logits[ix] = z.float().cpu().numpy()
    return logits

def probabilities(z):
    z = z-z.max(axis=1,keepdims=True)
    p = np.exp(z)
    return p/p.sum(axis=1,keepdims=True)

def write_submission(rows, docs, labels, logits, path):
    if logits.shape != (len(rows),len(labels)) or not np.isfinite(logits).all():
        raise ValueError('Invalid prediction array')
    prediction = {d['id']:[None]*len(d['text']) for d in docs}
    for row,pred in zip(rows,logits.argmax(1)):
        prediction[row['id']][row['sentence']] = labels[int(pred)]
    for d in docs:
        if len(prediction[d['id']])!=len(d['text']) or any(v not in labels for v in prediction[d['id']]):
            raise ValueError('Missing or invalid sentence prediction: '+d['id'])
    payload = json.dumps(prediction,ensure_ascii=False,indent=2)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',payload)
    return hashlib.sha256(payload.encode()).hexdigest()

## Training

In [6]:
def run_method(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    folder.mkdir(exist_ok=True)
    if (folder/'done.json').exists():
        previous = json.loads((folder/'done.json').read_text())
        if previous.get('run_id') != r['run_id']:
            raise ValueError('Settings changed; use a new OUT directory')
        print(f'GPU {gpu} | {method} already complete; reusing outputs',flush=True)
        return
    with (root/'data.pkl').open('rb') as f: data = cloudpickle.load(f)
    train,dev,test = [data['rows'][s] for s in ['train','dev','test']]
    gold = np.array([v['y'] for v in dev])
    soft = None
    if method=='distill':
        tm = make_model(teacher,c,r)
        ck = torch.load(root/teacher/'best.pt',map_location='cpu',weights_only=False)
        tm.load_state_dict(ck['weights'],strict=False)
        tm.to(device)
        soft = predict(tm,train,device,r)
        np.save(folder/'teacher_logits.npy',soft)
        del tm,ck
        gc.collect()
        torch.cuda.empty_cache()
    model = make_model(method,c,r).to(device)
    if hasattr(model,'adapter_config'):
        model.adapter_config.save_pretrained(folder)
    params = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
    body = [p for n,p in params if not n.startswith(('head.','domain.'))]
    head = [p for n,p in params if n.startswith(('head.','domain.'))]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body,lr=lr,initial_lr=lr),
                             dict(params=head,lr=3e-4,initial_lr=3e-4)],weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024)
    count = np.bincount([v['y'] for v in train],minlength=r['classes'])
    weight = 1/np.sqrt(np.maximum(count,1))
    weight = np.clip(weight/weight.mean(),.25,4.)
    weight = torch.tensor(weight,dtype=torch.float32,device=device)
    nsteps = math.ceil(len(train)/(r['micro']*r['accum']))
    total = methods[method]['epochs']*nsteps
    warmup = max(10,int(.06*total))
    best,stale,step,history = -1.,0,0,[]
    start = time.monotonic()
    total_params = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for _,p in params)
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} ({torch.cuda.get_device_name(device)}) | model on {next(model.parameters()).device} | {total_params:,} parameters | {trained:,} trainable',flush=True)
    stop = False
    for epoch in range(1,methods[method]['epochs']+1):
        order = np.random.default_rng(r['seed']+epoch).permutation(len(train))
        loss_sum,seen = 0.,0
        for at in range(0,len(order),r['micro']*r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at+r['micro']*r['accum']]
            factor = min(1.,(step+1)/warmup)*(.1+.9*.5*(1+math.cos(math.pi*min(1.,step/total))))
            for g in opt.param_groups: g['lr'] = g['initial_lr']*factor
            for offset in range(0,len(group),r['micro']):
                ix = group[offset:offset+r['micro']]
                x,s,e,d,p,y = collate(train,ix,device,r)
                with torch.autocast('cuda',dtype=torch.float16):
                    z = model(x,s,e,d,p)
                    ce = (F.cross_entropy(z.float(),y,reduction='none')*weight[y]).mean()
                    loss = ce
                    if soft is not None:
                        target = torch.from_numpy(soft[ix]).to(device)
                        kd = F.kl_div(F.log_softmax(z.float()/2.,-1),
                                      F.softmax(target/2.,-1),reduction='batchmean')*4.
                        loss = .5*ce+.5*kd
                    scaled = loss*(len(ix)/len(group))
                scaler.scale(scaled).backward()
                loss_sum += loss.detach().item()*len(ix)
                seen += len(ix)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _,p in params],1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            evaluate = step%r['eval_steps']==0 or at+len(group)>=len(order)
            if evaluate:
                logits = predict(model,dev,device,r)
                score = f1_score(gold,logits.argmax(1),labels=list(range(r['classes'])),
                                 average='macro',zero_division=0)
                improved = score>best
                stale = 0 if improved else stale+1
                if improved:
                    best = float(score)
                    weights = {n:p.detach().cpu().float() for n,p in params}
                    torch.save(dict(weights=weights,method=method,config=model.lm.c,
                                    epoch=epoch,step=step,score=best,teacher=teacher),folder/'best.tmp')
                    os.replace(folder/'best.tmp',folder/'best.pt')
                    np.save(folder/'dev_logits.npy',logits)
                row = dict(epoch=epoch,step=step,loss=loss_sum/max(seen,1),macro_f1=float(score))
                history.append(row)
                (folder/'history.json').write_text(json.dumps(history,indent=2))
                print(f'{method} | epoch {epoch} step {step} | loss {row["loss"]:.4f} | dev F1 {score:.4f}',flush=True)
                if stale>=r['patience']:
                    stop=True
                    break
        if stop: break
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(ck['weights'],strict=False)
    test_logits = predict(model,test,device,r)
    np.save(folder/'test_logits.npy',test_logits)
    sub = root/'submissions'
    sub.mkdir(exist_ok=True)
    write_submission(test,data['docs']['test'],data['labels'],test_logits,sub/f'submission_A3_{method}.zip')
    dev_logits = np.load(folder/'dev_logits.npy')
    pred = dev_logits.argmax(1)
    report = classification_report(gold,pred,labels=list(range(r['classes'])),
                                   target_names=data['labels'],zero_division=0,output_dict=True)
    (folder/'classification_report.json').write_text(json.dumps(report,indent=2))
    result = dict(method=method,run_id=r['run_id'],macro_f1=best,parameters=total_params,trainable=trained,
                  epoch=ck['epoch'],step=ck['step'],last_step=step,teacher=teacher,
                  stopping_reason='patience' if stop else 'epoch_limit',
                  minutes=(time.monotonic()-start)/60,
                  peak_gb=torch.cuda.max_memory_allocated(device)/2**30)
    for dom,name in enumerate(['CL','IT']):
        mask = np.array([v['domain']==dom for v in dev])
        result[name+'_macro_f1'] = float(f1_score(gold[mask],pred[mask],labels=list(range(r['classes'])),
                                                 average='macro',zero_division=0))
    (folder/'done.json').write_text(json.dumps(result,indent=2))

## Two-GPU training

In [7]:
def run_jobs(names, worker=run_method):
    pending, running, finished = list(names), {}, set()
    first = ['full','partial','lora','lorma','ia3','loha']
    while pending or running:
        for gpu,(process,path,method) in list(running.items()):
            status = process.poll()
            if status is not None:
                if status != 0:
                    raise subprocess.CalledProcessError(status,process.args)
                path.unlink()
                del running[gpu]
                finished.add(method)
                print(f'GPU {gpu} finished {method}',flush=True)
        for gpu in range(2):
            if gpu in running:
                continue
            ready = [m for m in pending if m!='distill' or set(first).issubset(finished)]
            if not ready:
                continue
            method = ready[0]
            teacher = None
            if method=='distill':
                teacher = max(first,key=lambda m:json.loads((OUT/m/'done.json').read_text())['macro_f1'])
            path = OUT/f'worker_{gpu}.pkl'
            with path.open('wb') as f:
                cloudpickle.dump((worker,(gpu,method,C,R,METHODS,teacher)),f)
            command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
            process = subprocess.Popen([sys.executable,'-u','-c',command,str(path)],env=os.environ.copy())
            running[gpu] = (process,path,method)
            pending.remove(method)
            print(f'GPU {gpu} starting {method} | PID {process.pid}',flush=True)
        if pending and not running:
            raise RuntimeError('Distillation requires the first six methods in this job queue')
        if running:
            time.sleep(.5)
run_jobs(list(METHODS))

GPU 0 starting full | PID 80
GPU 1 starting partial | PID 81
partial | GPU 1 (Tesla T4) | model on cuda:1 | 265,425,693 parameters | 77,644,573 trainable
full | GPU 0 (Tesla T4) | model on cuda:0 | 265,425,693 parameters | 265,425,693 trainable
partial | epoch 1 step 200 | loss 0.7100 | dev F1 0.1975
full | epoch 1 step 200 | loss 0.6854 | dev F1 0.2393
partial | epoch 1 step 400 | loss 0.6041 | dev F1 0.3194
partial | epoch 1 step 600 | loss 0.5469 | dev F1 0.3083
partial | epoch 1 step 800 | loss 0.5033 | dev F1 0.3494
full | epoch 1 step 400 | loss 0.5652 | dev F1 0.2996
partial | epoch 1 step 1000 | loss 0.4614 | dev F1 0.3569
partial | epoch 1 step 1053 | loss 0.4525 | dev F1 0.3339
full | epoch 1 step 600 | loss 0.4987 | dev F1 0.3319
partial | epoch 2 step 1200 | loss 0.1678 | dev F1 0.3385
partial | epoch 2 step 1400 | loss 0.1515 | dev F1 0.3459
full | epoch 1 step 800 | loss 0.4470 | dev F1 0.3578
partial | epoch 2 step 1600 | loss 0.1533 | dev F1 0.3572
partial | epoch 2 ste

In [8]:
teacher = json.loads((OUT/'distill'/'done.json').read_text())['teacher']
print('Distillation teacher:',teacher)

Distillation teacher: full


## Evaluation and submissions

In [9]:
gold = np.array([v['y'] for v in DATA['dev']])
results = [json.loads((OUT/m/'done.json').read_text()) for m in METHODS]
ranked = sorted(results,key=lambda v:v['macro_f1'],reverse=True)
members = [v['method'] for v in ranked[:3]]
dev_probs = np.stack([probabilities(np.load(OUT/m/'dev_logits.npy')) for m in members])
test_probs = np.stack([probabilities(np.load(OUT/m/'test_logits.npy')) for m in members])
candidates = [np.ones(len(members))/len(members),
              np.array([max(v['macro_f1'],1e-6) for v in ranked[:3]])]
candidates[1] /= candidates[1].sum()
scores = [f1_score(gold,np.einsum('m,mnc->nc',w,dev_probs).argmax(1),
                   labels=list(range(len(labels))),average='macro',zero_division=0) for w in candidates]
w = candidates[int(np.argmax(scores))]
ensemble = np.einsum('m,mnc->nc',w,test_probs)
selection = dict(members=members,weights=w.tolist(),dev_macro_f1=float(max(scores)))
(OUT/'ensemble.json').write_text(json.dumps(selection,indent=2))
results.append(dict(method='ensemble',macro_f1=float(max(scores)),members=members))
SUB = OUT/'submissions'
SUB.mkdir(exist_ok=True)
fingerprints = {}
for method in list(METHODS)+['ensemble']:
    logits = ensemble if method=='ensemble' else np.load(OUT/method/'test_logits.npy')
    digest = write_submission(DATA['test'],DOCS['test'],labels,logits,SUB/f'submission_A3_{method}.zip')
    if digest in fingerprints:
        print(f'Identical predictions: {method} and {fingerprints[digest]}')
    fingerprints[digest] = method
(OUT/'results.json').write_text(json.dumps(results,indent=2))
for v in sorted(results,key=lambda v:v['macro_f1'],reverse=True):
    print(f'{v["method"]:10s} dev Macro-F1 {v["macro_f1"]:.4f}')
print(f'Nine submission ZIPs: {SUB}')

ensemble   dev Macro-F1 0.3987
full       dev Macro-F1 0.3767
loha       dev Macro-F1 0.3673
lorma      dev Macro-F1 0.3673
partial    dev Macro-F1 0.3572
lora       dev Macro-F1 0.3534
ia3        dev Macro-F1 0.3511
prompt     dev Macro-F1 0.3356
distill    dev Macro-F1 0.2635
Nine submission ZIPs: /kaggle/working/phase3_a3_peft/submissions


In [10]:
with zipfile.ZipFile(OUT/'all_submissions_A3.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.glob('*.zip')): z.write(p,p.name)
    for name in ['results.json','ensemble.json','manifest.json']: z.write(OUT/name,name)
print(OUT/'all_submissions_A3.zip')

/kaggle/working/phase3_a3_peft/all_submissions_A3.zip


## Run notes